# Phase 2: Baseline CNN Classifier Development

This notebook trains a baseline ResNet18 model on the imbalanced dataset.

**Goals:**
1. Define a custom Dataset class to load images from `processed_data/`.
2. Fine-tune a pre-trained ResNet18 model for binary classification (Normal vs Defect).
3. Train on the raw, imbalanced data.
4. Evaluate performance (Precision, Recall, F1).

**Note:** Expect high Accuracy but potentially low Recall on defects due to imbalance.

In [ ]:
# 1. Imports & Setup
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Dataset
import os
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Constants
IMG_SIZE = 128 # Must match preprocessing
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 10

In [ ]:
# 2. Custom Dataset
class FabricDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        # 0 = Normal, 1 = Defect
        # Structure: root_dir/normal/*.png, root_dir/defect/*.png
        # OR root_dir/*.png (if flatten logic used - checking implementation in NB1)
        # NB1 implementation: 
        # processed_data/train_normal/*.png (Normal)
        # processed_data/train_defect/*.png (Defect)
        # processed_data/valid_mixed/normal/*.png
        # processed_data/valid_mixed/defect/*.png
        
        # Logic specifically for 'train' split which is separated
        if 'train' in root_dir:
            # It might be passed as 'processed_data'
            # We need to explicitly look into train_normal and train_defect
             # This part requires careful path handling based on how we call it
             pass
        
        # For simplicity, let's assume the user passes specific folders or we construct them
        # Let's write a flexible loader
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.float32)

def load_paths_from_folder(folder, label):
    paths = []
    labels = []
    if os.path.exists(folder):
        for f in os.listdir(folder):
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                paths.append(os.path.join(folder, f))
                labels.append(label)
    return paths, labels

# Transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) # [-1, 1] range
    ]),
    'val': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ]),
}

# Instantiate Datasets
# Train: Combine train_normal and train_defect
train_paths = []
train_labels = []

p, l = load_paths_from_folder('processed_data/train_normal', 0)
train_paths.extend(p); train_labels.extend(l)
p, l = load_paths_from_folder('processed_data/train_defect', 1)
train_paths.extend(p); train_labels.extend(l)

train_dataset = FabricDataset('processed_data') # Dummy root
train_dataset.image_paths = train_paths
train_dataset.labels = train_labels
train_dataset.transform = data_transforms['train']

# Valid
val_paths = []
val_labels = []
p, l = load_paths_from_folder('processed_data/valid_mixed/normal', 0)
val_paths.extend(p); val_labels.extend(l)
p, l = load_paths_from_folder('processed_data/valid_mixed/defect', 1)
val_paths.extend(p); val_labels.extend(l)

val_dataset = FabricDataset('processed_data')
val_dataset.image_paths = val_paths
val_dataset.labels = val_labels
val_dataset.transform = data_transforms['val']

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# 3. Model Definition
def get_model():
    model = models.resnet18(pretrained=True)
    # Modify final layer
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 1) # Binary classification output
    return model.to(device)

model = get_model()
criterion = nn.BCEWithLogitsLoss() # Computes sigmoid internally
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# 4. Training Loop
def train_model(model, train_loader, val_loader, epochs=EPOCHS):
    best_f1 = 0.0
    
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        print("-" * 10)
        
        # Train
        model.train()
        running_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Train Loss: {epoch_loss:.4f}")
        
        # Validate
        model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device).unsqueeze(1)
                outputs = model(inputs)
                preds = torch.sigmoid(outputs) > 0.5
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        accuracy = accuracy_score(all_labels, all_preds)
        precision = precision_score(all_labels, all_preds, zero_division=0)
        recall = recall_score(all_labels, all_preds, zero_division=0)
        f1 = f1_score(all_labels, all_preds, zero_division=0)
        
        print(f"Val Acc: {accuracy:.4f} | Prec: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
        
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), 'best_baseline_model.pth')
            print("Saved Best Model")
            
    return model

trained_model = train_model(model, train_loader, val_loader)

In [ ]:
# 5. Evaluation on Test Set
# Load test data
test_paths = []
test_labels = []
p, l = load_paths_from_folder('processed_data/test_mixed/normal', 0)
test_paths.extend(p); test_labels.extend(l)
p, l = load_paths_from_folder('processed_data/test_mixed/defect', 1)
test_paths.extend(p); test_labels.extend(l)

test_dataset = FabricDataset('processed_data')
test_dataset.image_paths = test_paths
test_dataset.labels = test_labels
test_dataset.transform = data_transforms['val']
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Evaluate
model.load_state_dict(torch.load('best_baseline_model.pth'))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = torch.sigmoid(outputs) > 0.5
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\nFinal Test Results (Baseline):")
print(f"Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
print(f"Precision: {precision_score(all_labels, all_preds, zero_division=0):.4f}")
print(f"Recall: {recall_score(all_labels, all_preds, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(all_labels, all_preds, zero_division=0):.4f}")

cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:")
print(cm)